In [1]:
import os
import warnings
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
import xarray as xr

# =========================================================
# 1. INPUT
# =========================================================
FILE_ER = "wk99_output_wave/wk_er_olr.nc"
FILE_MRG = "wk99_output_wave/wk_mrg_olr.nc"

OUTDIR = "calculation_output"
OUTFILE = os.path.join(OUTDIR, "Indeks ER MRG Double Side ER EWNS.xlsx")
os.makedirs(OUTDIR, exist_ok=True)

START_DATE = "1979-01-01"
END_DATE   = "2025-12-31"   # samakan semua sampai 2025

# =========================================================
# 2. BACA DATA
# =========================================================
erossby = xr.open_dataset(FILE_ER)
mrossby = xr.open_dataset(FILE_MRG)

# =========================================================
# 3. HITUNG INDEKS ER
# =========================================================
idxer1 = (
    erossby["olr_er"]
    .sel(lat=slice(3, 14), lon=slice(86, 101), time=slice(START_DATE, END_DATE))
    .mean(dim=("lat", "lon"))
)

idxer2 = (
    erossby["olr_er"]
    .sel(lat=slice(-16, -4), lon=slice(113, 128), time=slice(START_DATE, END_DATE))
    .mean(dim=("lat", "lon"))
)

idxer = idxer1 - idxer2

# =========================================================
# 4. HITUNG INDEKS MRG
# =========================================================
mrg1 = (
    mrossby["olr_mrg"]
    .sel(lat=slice(2, 9), lon=slice(97, 110), time=slice(START_DATE, END_DATE))
    .mean(dim=("lat", "lon"))
)

mrg2 = (
    mrossby["olr_mrg"]
    .sel(lat=slice(-6, -3), lon=slice(105, 117), time=slice(START_DATE, END_DATE))
    .mean(dim=("lat", "lon"))
)

idxmrg = mrg1 - mrg2

# =========================================================
# 5. SAMAKAN TIME ER DAN MRG
# =========================================================
idxer, idxmrg = xr.align(idxer, idxmrg, join="inner")

# cek panjang awal
print("Panjang idxer setelah align :", idxer.sizes["time"])
print("Panjang idxmrg setelah align:", idxmrg.sizes["time"])

# =========================================================
# 6. CENTRAL DIFFERENCE
#    time ikut dipotong [1:-1]
# =========================================================
time = pd.to_datetime(idxer["time"].values[1:-1])

idxercut = idxer.values[1:-1]
idxmrgcut = idxmrg.values[1:-1]

dtidxer = (idxer.values[2:] - idxer.values[:-2]) / 2.0
dtidxmrg = (idxmrg.values[2:] - idxmrg.values[:-2]) / 2.0

# =========================================================
# 7. NORMALISASI
# =========================================================
std_idxer = np.std(idxercut)
std_idxmrg = np.std(idxmrgcut)
std_dtidxer = np.std(dtidxer)
std_dtidxmrg = np.std(dtidxmrg)

if std_idxer == 0 or std_idxmrg == 0 or std_dtidxer == 0 or std_dtidxmrg == 0:
    raise ValueError("Ada standar deviasi bernilai 0, indeks tidak bisa dinormalisasi.")

idxercut = idxercut / std_idxer
idxmrgcut = idxmrgcut / std_idxmrg
dtidxer = dtidxer / std_dtidxer
dtidxmrg = dtidxmrg / std_dtidxmrg

# =========================================================
# 8. AMPLITUDO
# =========================================================
amper = np.sqrt(dtidxer**2 + idxercut**2)
ampmrg = np.sqrt(dtidxmrg**2 + idxmrgcut**2)

# =========================================================
# 9. SUDUT
# =========================================================
anger = np.rad2deg(np.arctan2(dtidxer, idxercut))
angmrg = np.rad2deg(np.arctan2(dtidxmrg, idxmrgcut))

# =========================================================
# 10. FASE (VECTORIZED)
#     P1 = kanan
#     P2 = bawah
#     P3 = kiri
#     P4 = atas
# =========================================================
pher = np.full(len(anger), np.nan)
phmrg = np.full(len(angmrg), np.nan)

# ER
pher[(anger <= 45) & (anger > -45)] = 1
pher[(anger <= -45) & (anger > -135)] = 2
pher[(anger <= -135) | (anger > 135)] = 3
pher[(anger <= 135) & (anger > 45)] = 4

# MRG
phmrg[(angmrg <= 45) & (angmrg > -45)] = 1
phmrg[(angmrg <= -45) & (angmrg > -135)] = 2
phmrg[(angmrg <= -135) | (angmrg > 135)] = 3
phmrg[(angmrg <= 135) & (angmrg > 45)] = 4

# ubah ke int
pher = pher.astype(int)
phmrg = phmrg.astype(int)

# =========================================================
# 11. CEK PANJANG ARRAY
# =========================================================
print("Panjang time    :", len(time))
print("Panjang phase ER:", len(pher))
print("Panjang amp ER  :", len(amper))
print("Panjang phase MRG:", len(phmrg))
print("Panjang amp MRG  :", len(ampmrg))

# =========================================================
# 12. DATAFRAME
# =========================================================
tglidxermrg = pd.DataFrame({
    "time": time,
    "phase er": pher,
    "amp er": amper,
    "phase mrg": phmrg,
    "amp mrg": ampmrg,
    "olr er": idxercut,
    "d(olr er)/dt": dtidxer,
    "olr mrg": idxmrgcut,
    "d(olr mrg)/dt": dtidxmrg,
})

# =========================================================
# 13. SAVE EXCEL
# =========================================================
with pd.ExcelWriter(OUTFILE, engine="openpyxl") as writer:
    tglidxermrg.to_excel(writer, sheet_name="EW", index=False)

print("Selesai.")
print("Output:", OUTFILE)
print(tglidxermrg.head())

Panjang idxer setelah align : 17167
Panjang idxmrg setelah align: 17167
Panjang time    : 17165
Panjang phase ER: 17165
Panjang amp ER  : 17165
Panjang phase MRG: 17165
Panjang amp MRG  : 17165
Selesai.
Output: calculation_output/Indeks ER MRG Double Side ER EWNS.xlsx
                 time  phase er    amp er  phase mrg   amp mrg    olr er  \
0 1979-01-02 12:00:00         1  0.065034          4  0.063121  0.058488   
1 1979-01-03 12:00:00         1  0.074037          1  0.088573  0.068819   
2 1979-01-04 12:00:00         1  0.078913          2  0.083367  0.076995   
3 1979-01-05 12:00:00         1  0.080571          3  0.067216  0.080542   
4 1979-01-06 12:00:00         1  0.079685          3  0.038237  0.078466   

   d(olr er)/dt   olr mrg  d(olr mrg)/dt  
0      0.028436  0.037671       0.050647  
1      0.027303  0.087939      -0.010577  
2      0.017296  0.018235      -0.081349  
3      0.002171 -0.061548      -0.027016  
4     -0.013887 -0.031411       0.021805  


In [2]:
import os
import warnings
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
import xarray as xr

# =========================================================
# 1. INPUT
# =========================================================
FILE_ER = "wk99_output_wave/wk_er_olr.nc"
FILE_MRG = "wk99_output_wave/wk_mrg_olr.nc"
FILE_KELVIN = "wk99_output_wave/wk_kelvin_olr.nc"

OUTDIR = "calculation_output"
OUTFILE = os.path.join(
    OUTDIR,
    "Indeks ER MRG Kelvin Double Side ER EWNS.xlsx"
)
os.makedirs(OUTDIR, exist_ok=True)

START_DATE = "1979-01-01"
END_DATE   = "2025-12-31"

# =========================================================
# 2. BACA DATA
# =========================================================
erossby = xr.open_dataset(FILE_ER)
mrossby = xr.open_dataset(FILE_MRG)
kelvin  = xr.open_dataset(FILE_KELVIN)

# =========================================================
# 3. HITUNG INDEKS ER
# =========================================================
idxer1 = (
    erossby["olr_er"]
    .sel(lat=slice(3, 14), lon=slice(86, 101), time=slice(START_DATE, END_DATE))
    .mean(dim=("lat", "lon"))
)

idxer2 = (
    erossby["olr_er"]
    .sel(lat=slice(-16, -4), lon=slice(113, 128), time=slice(START_DATE, END_DATE))
    .mean(dim=("lat", "lon"))
)

idxer = idxer1 - idxer2

# =========================================================
# 4. HITUNG INDEKS MRG
# =========================================================
mrg1 = (
    mrossby["olr_mrg"]
    .sel(lat=slice(2, 9), lon=slice(97, 110), time=slice(START_DATE, END_DATE))
    .mean(dim=("lat", "lon"))
)

mrg2 = (
    mrossby["olr_mrg"]
    .sel(lat=slice(-6, -3), lon=slice(105, 117), time=slice(START_DATE, END_DATE))
    .mean(dim=("lat", "lon"))
)

idxmrg = mrg1 - mrg2

# =========================================================
# 5. HITUNG INDEKS KELVIN
# =========================================================
# Catatan:
# Kelvin biasanya simetris dekat ekuator dan propagasi ke timur.
# Di sini indeks dibuat dari rata-rata OLR Kelvin pada wilayah ekuator
# Maritime Continent. Silakan sesuaikan box jika mengikuti paper tertentu.

idxkelvin = (
    kelvin["olr_kelvin"]
    .sel(lat=slice(-5, 5), lon=slice(100, 120), time=slice(START_DATE, END_DATE))
    .mean(dim=("lat", "lon"))
)

# =========================================================
# 6. SAMAKAN TIME ER, MRG, KELVIN
# =========================================================
idxer, idxmrg, idxkelvin = xr.align(idxer, idxmrg, idxkelvin, join="inner")

print("Panjang idxer setelah align    :", idxer.sizes["time"])
print("Panjang idxmrg setelah align   :", idxmrg.sizes["time"])
print("Panjang idxkelvin setelah align:", idxkelvin.sizes["time"])

# =========================================================
# 7. CENTRAL DIFFERENCE
# =========================================================
time = pd.to_datetime(idxer["time"].values[1:-1])

idxercut = idxer.values[1:-1]
idxmrgcut = idxmrg.values[1:-1]
idxkelvincut = idxkelvin.values[1:-1]

dtidxer = (idxer.values[2:] - idxer.values[:-2]) / 2.0
dtidxmrg = (idxmrg.values[2:] - idxmrg.values[:-2]) / 2.0
dtidxkelvin = (idxkelvin.values[2:] - idxkelvin.values[:-2]) / 2.0

# =========================================================
# 8. NORMALISASI
# =========================================================
std_idxer = np.std(idxercut)
std_idxmrg = np.std(idxmrgcut)
std_idxkelvin = np.std(idxkelvincut)

std_dtidxer = np.std(dtidxer)
std_dtidxmrg = np.std(dtidxmrg)
std_dtidxkelvin = np.std(dtidxkelvin)

if (
    std_idxer == 0 or std_idxmrg == 0 or std_idxkelvin == 0 or
    std_dtidxer == 0 or std_dtidxmrg == 0 or std_dtidxkelvin == 0
):
    raise ValueError("Ada standar deviasi bernilai 0, indeks tidak bisa dinormalisasi.")

idxercut = idxercut / std_idxer
idxmrgcut = idxmrgcut / std_idxmrg
idxkelvincut = idxkelvincut / std_idxkelvin

dtidxer = dtidxer / std_dtidxer
dtidxmrg = dtidxmrg / std_dtidxmrg
dtidxkelvin = dtidxkelvin / std_dtidxkelvin

# =========================================================
# 9. AMPLITUDO
# =========================================================
amper = np.sqrt(dtidxer**2 + idxercut**2)
ampmrg = np.sqrt(dtidxmrg**2 + idxmrgcut**2)
ampkelvin = np.sqrt(dtidxkelvin**2 + idxkelvincut**2)

# =========================================================
# 10. SUDUT
# =========================================================
anger = np.rad2deg(np.arctan2(dtidxer, idxercut))
angmrg = np.rad2deg(np.arctan2(dtidxmrg, idxmrgcut))
angkelvin = np.rad2deg(np.arctan2(dtidxkelvin, idxkelvincut))

# =========================================================
# 11. FASE
#     P1 = kanan
#     P2 = bawah
#     P3 = kiri
#     P4 = atas
# =========================================================
def angle_to_phase(angle):
    phase = np.full(len(angle), np.nan)

    phase[(angle <= 45) & (angle > -45)] = 1
    phase[(angle <= -45) & (angle > -135)] = 2
    phase[(angle <= -135) | (angle > 135)] = 3
    phase[(angle <= 135) & (angle > 45)] = 4

    return phase.astype(int)

pher = angle_to_phase(anger)
phmrg = angle_to_phase(angmrg)
phkelvin = angle_to_phase(angkelvin)

# =========================================================
# 12. CEK PANJANG ARRAY
# =========================================================
print("Panjang time        :", len(time))

print("Panjang phase ER    :", len(pher))
print("Panjang amp ER      :", len(amper))

print("Panjang phase MRG   :", len(phmrg))
print("Panjang amp MRG     :", len(ampmrg))

print("Panjang phase Kelvin:", len(phkelvin))
print("Panjang amp Kelvin  :", len(ampkelvin))

# =========================================================
# 13. DATAFRAME
# =========================================================
tglidxermrg = pd.DataFrame({
    "time": time,

    "phase er": pher,
    "amp er": amper,
    "olr er": idxercut,
    "d(olr er)/dt": dtidxer,

    "phase mrg": phmrg,
    "amp mrg": ampmrg,
    "olr mrg": idxmrgcut,
    "d(olr mrg)/dt": dtidxmrg,

    "phase kelvin": phkelvin,
    "amp kelvin": ampkelvin,
    "olr kelvin": idxkelvincut,
    "d(olr kelvin)/dt": dtidxkelvin,
})

# =========================================================
# 14. SAVE EXCEL
# =========================================================
with pd.ExcelWriter(OUTFILE, engine="openpyxl") as writer:
    tglidxermrg.to_excel(writer, sheet_name="EW", index=False)

print("Selesai.")
print("Output:", OUTFILE)
print(tglidxermrg.head())

Panjang idxer setelah align    : 17167
Panjang idxmrg setelah align   : 17167
Panjang idxkelvin setelah align: 17167
Panjang time        : 17165
Panjang phase ER    : 17165
Panjang amp ER      : 17165
Panjang phase MRG   : 17165
Panjang amp MRG     : 17165
Panjang phase Kelvin: 17165
Panjang amp Kelvin  : 17165
Selesai.
Output: calculation_output/Indeks ER MRG Kelvin Double Side ER EWNS.xlsx
                 time  phase er    amp er    olr er  d(olr er)/dt  phase mrg  \
0 1979-01-02 12:00:00         1  0.065034  0.058488      0.028436          4   
1 1979-01-03 12:00:00         1  0.074037  0.068819      0.027303          1   
2 1979-01-04 12:00:00         1  0.078913  0.076995      0.017296          2   
3 1979-01-05 12:00:00         1  0.080571  0.080542      0.002171          3   
4 1979-01-06 12:00:00         1  0.079685  0.078466     -0.013887          3   

    amp mrg   olr mrg  d(olr mrg)/dt  phase kelvin  amp kelvin  olr kelvin  \
0  0.063121  0.037671       0.050647          